# Required imports
Always run

In [1]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

import librosa
from scipy.fftpack import dct

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten, GaussianNoise, Reshape
from tensorflow.keras.layers import GlobalAveragePooling1D

# Load feature

In [2]:
# Load feature
X_train = np.load("features/X_train.npy")
X_test  = np.load("features/X_test.npy")
y_train = np.load("features/y_train.npy")
y_test  = np.load("features/y_test.npy")
labels   = np.load("features/labels.npy")

print("X_train shape:", X_train.shape, "dtype:", X_train.dtype)
print("X_test  shape:", X_test.shape,  "dtype:", X_test.dtype)
print("y_train shape:", y_train.shape, "dtype:", y_train.dtype)
print("y_test  shape:", y_test.shape,  "dtype:", y_test.dtype)
print("labels:", labels)

X_train shape: (2615, 49, 32) dtype: int8
X_test  shape: (641, 49, 32) dtype: int8
y_train shape: (2615, 5) dtype: int64
y_test  shape: (641, 5) dtype: int64
labels: ['noise' 'unknown' 'heynano' 'on' 'off']


In [ ]:
# Create an array for the test sample in C
prova = X_test[402].flatten()

# write on file the entire feature tensor of prova test sample
with open("features/test_features.txt", "w") as f:
    f.write("{ ")
    for value in prova:
        f.write(f"{value}, ")
    f.write("};")

# Load tflite model

In [7]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path="models/best.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

num_sample = 402

input_data = X_test[num_sample]
input_data = input_data.flatten()
input_data = np.expand_dims(input_data, axis=0)
print(input_data.shape, input_data.dtype)

interpreter.set_tensor(input_details[0]['index'], input_data)

interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])
out_scale, out_zero_point = output_details[0]['quantization']

if out_scale > 0:
    output_data = (output_data.astype(np.float32) - out_zero_point) * out_scale

print("Output:", output_data)
print("Expected:", y_test[num_sample])

(1, 1568) int8
Output: [[0.         0.         0.         0.99609375 0.        ]]
Expected: [0 0 0 1 0]
